# Data cleaning

**Missing data**:
- Iterative imputer (+/- indicator)
- KNN imputer (+/- indicator)

**Feature encoding**
- One-hot
- Target

## 1. Notebook setup

### 1.1. Imports

In [ ]:
# Standard library
import json

# Third party
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, LabelEncoder, StandardScaler

In [2]:
SAMPLE = 0.10      # Fraction of data to use for training
IMPUTE = True      # True = do imputation, False = load imputation results from disk
KNN_NEIGHBORS = 5  # Number of neighbors to use for KNN imputation
CV_SCORE_FOLDS = 5 # Number of cross-validation folds to use for scoring

### 1.3. Data loading

#### 1.3.1. Training data

In [ ]:
# Load raw data
train_df = pd.read_csv('https://media.githubusercontent.com/media/gperdrizet/fullstack-2605/refs/heads/main/data/student-health-risk-train.csv')

# Take sample
n = int(len(train_df) * SAMPLE)
train_df = train_df.sample(n=n)

# Preserve label column
training_label = train_df['health_condition']

# Remove id and label columns
train_df.drop(['id', 'health_condition'], axis=1, inplace=True)
train_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 69008 entries, 673226 to 492579
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   sleep_duration           61385 non-null  float64
 1   heart_rate               68196 non-null  float64
 2   bmi                      67534 non-null  float64
 3   calorie_expenditure      63648 non-null  float64
 4   step_count               67660 non-null  float64
 5   exercise_duration        68315 non-null  float64
 6   water_intake             64660 non-null  float64
 7   diet_type                68301 non-null  object 
 8   stress_level             60773 non-null  object 
 9   sleep_quality            63106 non-null  object 
 10  physical_activity_level  65336 non-null  object 
 11  smoking_alcohol          66102 non-null  object 
 12  gender                   66868 non-null  object 
dtypes: float64(7), object(6)
memory usage: 7.4+ MB


#### 1.3.2. Testing data

In [ ]:
# Load raw data
test_df = pd.read_csv('https://media.githubusercontent.com/media/gperdrizet/fullstack-2605/refs/heads/main/data/student-health-risk-test.csv')

# Preserve id column
testing_ids = test_df['id']

# Drop id column
test_df.drop('id', axis=1, inplace=True)
test_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   sleep_duration           263182 non-null  float64
 1   heart_rate               292396 non-null  float64
 2   bmi                      289797 non-null  float64
 3   calorie_expenditure      273101 non-null  float64
 4   step_count               289789 non-null  float64
 5   exercise_duration        292795 non-null  float64
 6   water_intake             277120 non-null  float64
 7   diet_type                292795 non-null  object 
 8   stress_level             260263 non-null  object 
 9   sleep_quality            270754 non-null  object 
 10  physical_activity_level  280058 non-null  object 
 11  smoking_alcohol          283504 non-null  object 
 12  gender                   286593 non-null  object 
dtypes: float64(7), object(6)
memory usage: 29.3+ MB


#### 1.3.3. Dataset metadata

Load feature definitions from disk. Treat our one high cardinality discrete feature `step_count` as continuous.

In [5]:
# Load dataset metadata
with open("../data/schema.json", "r") as f:
    metadata = json.load(f)

# Extract feature lists
features = metadata['features']
categorical_features = metadata['categorical_features']
continuous_features = metadata['continuous_features']
high_cardinality_feature = metadata['high_cardinality_feature']
label = metadata['label']

# Treat 'step_count' like a continuous feature
continuous_features += [high_cardinality_feature]

print(f'Categorical features: {", ".join(categorical_features)}')
print(f'Continuous features: {", ".join(continuous_features)}')
print(f'Label: {label}')

Categorical features: diet_type, stress_level, sleep_quality, physical_activity_level, smoking_alcohol, gender
Continuous features: sleep_duration, heart_rate, bmi, calorie_expenditure, step_count, exercise_duration, water_intake, step_count
Label: health_condition


## 2. Functions

### 2.1. Imputation

#### 2.1.1. Standard scaler

In [ ]:
def standard_scaler(
        train_df: pd.DataFrame,
        test_df: pd.DataFrame,
        features: list[str]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Standardize continuous features in the given DataFrames using Scikit-learn's StandardScaler.

    Args:
        train_df (pd.DataFrame): The training DataFrame.
        test_df (pd.DataFrame): The testing DataFrame.
        features (list): List of continuous features to standardize.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: The training and testing DataFrames with standardized continuous features.
    '''

    scaler = StandardScaler()
    scaler.fit(train_df[features])
    train_df[features] = scaler.transform(train_df[features])
    test_df[features] = scaler.transform(test_df[features])

    return train_df, test_df, scaler

#### 2.1.2. Missing indicator feature

In [ ]:
def add_missing_indicator(
        train_df: pd.DataFrame,
        test_df: pd.DataFrame,
        features: list
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Add missing value indicator features to the given DataFrames.

    Args:
        train_df (pd.DataFrame): The training DataFrame.
        test_df (pd.DataFrame): The testing DataFrame.
        features (list): List of features to create missing value indicators for.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: The training and testing DataFrames with added missing value indicator features.
    '''

    # Create column names for the new indicator features
    indicator_features = [f'{feature}_missing' for feature in features]

    # Create indicator features from boolean mask
    training_missing = train_df[features].isna().astype('int')
    training_missing.columns = indicator_features

    testing_missing = test_df[features].isna().astype('int')
    testing_missing.columns = indicator_features

    # Add indicator features to DataFrames
    train_df = pd.concat(
        [train_df.reset_index(drop=True),
        training_missing.reset_index(drop=True)],
        axis=1
    )

    test_df = pd.concat(
        [test_df.reset_index(drop=True),
        testing_missing.reset_index(drop=True)],
        axis=1
    )

    return train_df, test_df

#### 2.1.3. KNN imputer

In [ ]:
def knn_imputer(
        train_df: pd.DataFrame,
        test_df: pd.DataFrame,
        features: list[str],
        n_neighbors: int = 5,
        indicator: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Impute missing values in the given features using K-Nearest Neighbors (KNN) imputer.

    Args:
        train_df (pd.DataFrame): The training data with missing values.
        test_df (pd.DataFrame): The testing data with missing values.
        features (list[str]): The list of features to impute.
        n_neighbors (int, optional): Number of neighbors to use for imputation. Defaults to 5.
        indicator (bool, optional): Whether to add a boolean indicator for missing values. Defaults to False.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: The training and testing features with missing values imputed.
    '''

    # Scale the features before imputation
    train_df, test_df, scaler = standard_scaler(train_df, test_df, features)

    # Create boolean indicator for missing values if requested
    if indicator:
        train_df, test_df = add_missing_indicator(train_df, test_df, features)

    # Create the imputer
    imputer = KNNImputer(n_neighbors=n_neighbors, weights='distance')

    # Impute both datasets
    train_df = pd.DataFrame(
        imputer.fit_transform(train_df),
        columns=imputer.get_feature_names_out()
    )

    test_df = pd.DataFrame(
        imputer.transform(test_df),
        columns=imputer.get_feature_names_out()
    )

    # Unscale the features after imputation
    train_df[features] = scaler.inverse_transform(train_df[features])
    test_df[features] = scaler.inverse_transform(test_df[features])

    return train_df, test_df

#### 2.1.4. Iterative imputer

In [ ]:
def iterative_imputer(
        train_df: pd.DataFrame,
        test_df: pd.DataFrame,
        features: list[str],
        indicator: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Impute missing values in the given features using Iterative Imputer.

    Args:
        train_df (pd.DataFrame): The training data with missing values.
        test_df (pd.DataFrame): The testing data with missing values.
        features (list[str]): The list of features to impute.
        n_neighbors (int, optional): Number of neighbors to use for imputation. Defaults to 5.
        indicator (bool, optional): Whether to add a boolean indicator for missing values. Defaults to False.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: The training and testing features with missing values imputed.
    '''

    # Scale the features before imputation
    train_df, test_df, scaler = standard_scaler(train_df, test_df, features)

    # Create boolean indicator for missing values if requested
    if indicator:
        train_df, test_df = add_missing_indicator(train_df, test_df, features)

    # Create the imputer
    imputer = IterativeImputer()

    # Impute both datasets
    train_df = pd.DataFrame(
        imputer.fit_transform(train_df),
        columns=imputer.get_feature_names_out()
    )

    test_df = pd.DataFrame(
        imputer.transform(test_df),
        columns=imputer.get_feature_names_out()
    )

    # Unscale the features after imputation
    train_df[features] = scaler.inverse_transform(train_df[features])
    test_df[features] = scaler.inverse_transform(test_df[features])

    return train_df, test_df

### 2.2. Encoding

#### 2.2.1. One-hot encoder

In [ ]:
def one_hot_encoder(
        train_df: pd.DataFrame,
        test_df: pd.DataFrame,
        features: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Encodes features with Scikit-learn's one-hot-encoder.

    Args:
        train_df (pd.DataFrame): The training data to encode.
        test_df (pd.DataFrame): The testing data to encode.
        features (list[str]): The list of features to encode.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: The training and testing features with encoded features.
    '''
    
    # Scikit-learn one-hot encoder
    feature_encoder = OneHotEncoder(sparse_output=False, drop='first')

    # Adapt to training data categorical features
    result = feature_encoder.fit(train_df[features])

    # Transform training and testing data
    encoded_train_features = feature_encoder.transform(train_df[features])
    encoded_test_features = feature_encoder.transform(test_df[features])
    
    # Convert encoded features to DataFrame
    encoded_train_df = pd.DataFrame(encoded_train_features, columns=feature_encoder.get_feature_names_out())
    encoded_test_df = pd.DataFrame(encoded_test_features, columns=feature_encoder.get_feature_names_out())

    # Replace original categorical features with encoded features
    train_df = pd.concat(
        [train_df.drop(columns=features).reset_index(drop=True),
        encoded_train_df.reset_index(drop=True)],
        axis=1
    )

    test_df = pd.concat(
        [test_df.drop(columns=features).reset_index(drop=True),
        encoded_test_df.reset_index(drop=True)],
        axis=1
    )

    # Update feature list
    features = train_df.columns

    train_df.info()
    

#### 2.2.2. Target encoder

In [ ]:
def target_encoder(
        train_df: pd.DataFrame,
        test_df: pd.DataFrame,
        features: list[str],
        smooth: float | str = 'auto'
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Encodes features with Scikit-learn's target encoder.

    Args:
        train_df (pd.DataFrame): The training data to encode.
        test_df (pd.DataFrame): The testing data to encode.
        features (list[str]): The list of features to encode.
        smooth (float|str): Amount of mixing between category and global target mean.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: The training and testing features with encoded features.
    '''
    
    # Scikit-learn target encoder
    feature_encoder = TargetEncoder(smooth=smooth)

    # Adapt to training data categorical features
    result = feature_encoder.fit(train_df[features])

    # Transform training and testing data
    encoded_train_features = feature_encoder.transform(train_df[features])
    encoded_test_features = feature_encoder.transform(test_df[features])
    
    # Convert encoded features to DataFrame
    encoded_train_df = pd.DataFrame(encoded_train_features, columns=feature_encoder.get_feature_names_out())
    encoded_test_df = pd.DataFrame(encoded_test_features, columns=feature_encoder.get_feature_names_out())

    # Replace original categorical features with encoded features
    train_df = pd.concat(
        [train_df.drop(columns=features).reset_index(drop=True),
        encoded_train_df.reset_index(drop=True)],
        axis=1
    )

    test_df = pd.concat(
        [test_df.drop(columns=features).reset_index(drop=True),
        encoded_test_df.reset_index(drop=True)],
        axis=1
    )

    # Update feature list
    features = train_df.columns

    train_df.info()

#### 2.2.3. Label encoder

In [ ]:
def encode_label(label: pd.DataSeries) -> tuple[pd.DataSeries, LabelEncoder]:
    '''Encodes the training label with Scikit-learn's label encoder. Returns the encoded
    label as a series and the encoder.

    Args:
        label (pd.DataSeries): The training label column.

    Returns:
        tuple[pd.DataFrame, LabelEncoder]: The encoded labels and the LabelEncoder instance.
    '''

    
    # Scikit-learn label encoder
    label_encoder = LabelEncoder()

    # Convert string training labels to integer representation
    label = label_encoder.fit_transform(label)
    
    return label, label_encoder

### 3. Experiments

#### 3.1. Imputation x encoding strategy

In [ ]:
conditions = {
    'One-hot, KNN': {},
    'One-hot, iterative': {},
    'Target, KNN': {},
    'Target, iterative': {}
}